In [ ]:
from src.config import load_config,save_resolved_config
print(load_config)
print(type(load_config))

<function load_config at 0x0000027AFDAF5F30>
<class 'function'>


In [ ]:
from src.config import load_config, save_resolved_config
from pathlib import Path
cfg = load_config()

# Print some values
print("=== DATA ===")
print("Target column:", cfg.data.data_schema.target_col)
print("Context length:", cfg.data.windows.context_length)
print("Horizon:", cfg.data.windows.horizon)

print("\n=== MODEL ===")
print("Model name:", cfg.model.name)
print("Distribution:", cfg.model.head.option.distribution)

print("\n=== TRAIN ===")
print("Run name:", cfg.train.run.name)
print("Physics enabled:", cfg.train.physics.enabled)

print("\n=== EVAL ===")
print("Metrics:", cfg.eval.metrics.point.list)

# Save resolved config (test reproducibility)
out_path = Path("runs/test_config_resolved.yaml")
save_resolved_config(cfg=cfg, out_path=out_path)

print(f"\nResolved config saved to: {out_path.resolve()}")

=== DATA ===
Target column: load_mw
Context length: 168
Horizon: 24

=== MODEL ===
Model name: transformer_forecaster
Distribution: gaussian

=== TRAIN ===
Run name: exp_transformer_gaussian_v0
Physics enabled: True

=== EVAL ===
Metrics: ['mae', 'rmse']

Resolved config saved to: C:\Users\Asees\Documents\Project\physics-constrained-prob-load-forecast\runs\test_config_resolved.yaml


Testing ingest.py

In [ ]:
from src.config import load_config
from src.data.ingest import load_native_load_canonical

cfg = load_config()
df, stats = load_native_load_canonical(cfg.data)

print(df.head())
print(df.columns)
print(stats[0])

            timestamp        COAST         EAST        FWEST       NORTH  \
0 2019-01-01 01:00:00  9783.594839  1264.194059  3164.720629  827.759428   
1 2019-01-01 02:00:00  9726.062326  1270.494294  3178.950849  830.256302   
2 2019-01-01 03:00:00  9654.373521  1269.504203  3200.881160  831.505553   
3 2019-01-01 04:00:00  9631.735303  1243.168422  3221.963116  839.689838   
4 2019-01-01 05:00:00  9698.889228  1309.572865  3240.634870  848.672153   

          NCENT        SOUTH        SCENT         WEST         ERCOT  
0  11697.766093  3066.891146  5993.974289  1282.542956  37081.443439  
1  11787.936174  3140.383209  6029.773970  1295.132811  37258.989933  
2  11861.628382  3137.721342  6043.348512  1301.225133  37300.187809  
3  11989.105118  3105.497101  6086.408464  1305.976101  37423.543464  
4  12219.106347  3078.256356  6180.939962  1319.140512  37895.212293  
Index(['timestamp', 'COAST', 'EAST', 'FWEST', 'NORTH', 'NCENT', 'SOUTH',
       'SCENT', 'WEST', 'ERCOT'],
      dtyp

In [ ]:
print(df.shape)
print(df.columns)
print(stats[:2])
print(stats[-1])
print(df["timestamp"].min(), df["timestamp"].max())

(58781, 10)
Index(['timestamp', 'COAST', 'EAST', 'FWEST', 'NORTH', 'NCENT', 'SOUTH',
       'SCENT', 'WEST', 'ERCOT'],
      dtype='object')
[IngestYearStats(year=2019, zip_path=WindowsPath('data/raw/2019/Native_Load_2019.zip'), xlsx_file='Native_Load_2019.xlsx', chosen_sheet='Native Load Report', rows_read=8760, rows_after_basic_clean=8394, dropped_bad_timestamps=366, dropped_missing_target=0), IngestYearStats(year=2020, zip_path=WindowsPath('data/raw/2020/Native_Load_2020.zip'), xlsx_file='Native_Load_2020.xlsx', chosen_sheet='Native Load Report', rows_read=8784, rows_after_basic_clean=8417, dropped_bad_timestamps=367, dropped_missing_target=0)]
IngestYearStats(year=2025, zip_path=WindowsPath('data/raw/2025/Native_Load_2025.zip'), xlsx_file='Native_Load_2025.xlsx', chosen_sheet='Sheet1', rows_read=8760, rows_after_basic_clean=8394, dropped_bad_timestamps=366, dropped_missing_target=0)
2019-01-01 01:00:00 2025-12-31 23:00:00


In [ ]:
# features.py
from src.config import load_config
from src.data.ingest import load_native_load_canonical
from src.data.features import build_processed_frames

cfg = load_config()

canon, ingest_stats = load_native_load_canonical(cfg.data)
system_df, zones_df, stats = build_processed_frames(canon, cfg.data)

print(system_df.head())
print(stats)

if zones_df is not None:
    print("zones_df columns:", zones_df.columns.tolist())
    print(zones_df.head())

            timestamp       load_mw
0 2019-01-01 01:00:00  37081.443439
1 2019-01-01 02:00:00  37258.989933
2 2019-01-01 03:00:00  37300.187809
3 2019-01-01 04:00:00  37423.543464
4 2019-01-01 05:00:00  37895.212293
CleanStats(rows_in=58781, rows_after_drop_bad_timestamps=58781, rows_after_dedupe=50387, rows_after_missing_policy=50387, duplicates_dropped=8394, missing_target_dropped=0, missing_hours_estimated=10980, start_timestamp='2019-01-01 01:00:00', end_timestamp='2025-12-31 23:00:00', freq='h', timezone_note='Stored as naive local time; interpret as America/Chicago', missing_policy='drop', hour_convention_note='ERCOT Hour Ending (stored naive local time)', rows_out_system=50387, rows_out_zones=50387, save_zones=True)
zones_df columns: ['timestamp', 'COAST', 'EAST', 'FWEST', 'NORTH', 'NCENT', 'SOUTH', 'SCENT', 'WEST', 'ERCOT']
            timestamp        COAST         EAST        FWEST       NORTH  \
0 2019-01-01 01:00:00  9783.594839  1264.194059  3164.720629  827.759428   
1 20

In [ ]:
system_df[system_df["timestamp"].dt.year == 2024].head()

,timestamp,load_mw


In [ ]:
import pandas as pd
df = pd.read_parquet("data/processed/ercot_hourly_v0.parquet")
df.head(), df.shape

(            timestamp       load_mw  hour_sin  hour_cos   dow_sin  dow_cos  \
 0 2019-01-01 01:00:00  37081.443439  0.258819  0.965926  0.781832  0.62349   
 1 2019-01-01 02:00:00  37258.989933  0.500000  0.866025  0.781832  0.62349   
 2 2019-01-01 03:00:00  37300.187809  0.707107  0.707107  0.781832  0.62349   
 3 2019-01-01 04:00:00  37423.543464  0.866025  0.500000  0.781832  0.62349   
 4 2019-01-01 05:00:00  37895.212293  0.965926  0.258819  0.781832  0.62349   
 
    month_sin  month_cos  is_weekend  is_holiday  
 0        0.0        1.0           0           1  
 1        0.0        1.0           0           1  
 2        0.0        1.0           0           1  
 3        0.0        1.0           0           1  
 4        0.0        1.0           0           1  ,
 (61367, 10))

In [ ]:
# testing the data loaders

from src.data.datasets import make_dataloaders

# Build loaders
train_loader, val_loader, test_loader = make_dataloaders(
    source="system",   # or "zones"
    batch_size=4,
    return_meta=True   # IMPORTANT to see timestamps
)
X, y, meta = next(iter(train_loader))
print(X.shape, y.shape)
print(meta.keys())
for k,v in meta.items():
    print(k, v[0])

torch.Size([4, 168, 1]) torch.Size([4, 24])
dict_keys(['t_end', 'x_start', 'x_end', 'y_start', 'y_end', 't_end_idx', 'x0', 'x1', 'y0', 'y1'])
t_end 2022-08-10T07:00:00.000000000
x_start 2022-08-03T08:00:00.000000000
x_end 2022-08-10T07:00:00.000000000
y_start 2022-08-10T08:00:00.000000000
y_end 2022-08-11T07:00:00.000000000
t_end_idx tensor(31614)
x0 tensor(31447)
x1 tensor(31615)
y0 tensor(31615)
y1 tensor(31639)


In [ ]:
import pandas as pd

# meta values are strings; convert to timestamps
xs = pd.Timestamp(meta["x_start"][0])
xe = pd.Timestamp(meta["x_end"][0])

# the expected number of hourly points in inclusive range is 168
expected = int(((xe - xs).total_seconds() // 3600) + 1)
print("Expected X length:", expected, "Actual:", X.shape[1])

Expected X length: 168 Actual: 168


In [ ]:
import pandas as pd
import numpy as np

# Load processed data
df = pd.read_parquet("data/processed/ercot_hourly_v0.parquet")

# Get window indices from meta
x0 = int(meta["x0"][0])
x1 = int(meta["x1"][0])  # exclusive

# Extract timestamps for that window
ts_window = pd.to_datetime(df["timestamp"].iloc[x0:x1].values)

# Compute hour differences
dh = (ts_window[1:] - ts_window[:-1]) / np.timedelta64(1, "h")

print("Unique hour steps in window:", np.unique(dh))
print("Count of non-1h steps:", np.sum(dh != 1))

Unique hour steps in window: [1.]
Count of non-1h steps: 0


In [ ]:
import torch
import numpy as np
import pandas as pd

from src.data.datasets import make_dataloaders

# Choose source: "system" or "zones"
train_loader, val_loader, test_loader = make_dataloaders(
    source="system",  # change to "zones" to inspect zones
    batch_size=2,
    return_meta=True
)

X, y, meta = next(iter(train_loader))

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeature dimension:", X.shape[-1])
print("First batch sample - first 5 timesteps:")
print(X[0, :5])

print("\nFirst batch target:")
print(y[0])

print("\nMeta (first sample):")
print({k: v[0] for k, v in meta.items()})

# Check for NaNs
print("\nAny NaNs in X?", torch.isnan(X).any().item())
print("Any NaNs in y?", torch.isnan(y).any().item())

X shape: torch.Size([2, 168, 5])
y shape: torch.Size([2, 24])

Feature dimension: 5
First batch sample - first 5 timesteps:
tensor([[ 6.0996e+04,  2.5882e-01, -9.6593e-01, -9.7493e-01, -2.2252e-01],
        [ 6.5527e+04,  1.2246e-16, -1.0000e+00, -9.7493e-01, -2.2252e-01],
        [ 6.9457e+04, -2.5882e-01, -9.6593e-01, -9.7493e-01, -2.2252e-01],
        [ 7.2752e+04, -5.0000e-01, -8.6603e-01, -9.7493e-01, -2.2252e-01],
        [ 7.5369e+04, -7.0711e-01, -7.0711e-01, -9.7493e-01, -2.2252e-01]])

First batch target:
tensor([56550.1211, 60656.3164, 64203.2656, 67006.1250, 69204.3750, 70897.8984,
        71535.4453, 70662.7266, 67898.1641, 64796.3477, 62203.9453, 59588.3320,
        56540.2031, 56540.2031, 50643.7305, 48081.9688, 45905.9570, 44309.9766,
        43206.3086, 42631.0742, 42613.0391, 42787.9453, 44801.0820, 48544.1992])

Meta (first sample):
{'t_end': '2023-09-30T10:00:00.000000000', 'x_start': '2023-09-23T11:00:00.000000000', 'x_end': '2023-09-30T10:00:00.000000000', 'y_star

In [ ]:
import pandas as pd
df = pd.read_parquet("data/processed/ercot_hourly_v0.parquet")
print(df.columns.tolist())

['timestamp', 'load_mw', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'is_holiday']


In [ ]:
# testing the baseline

In [ ]:
from src.config import load_config
from src.data.datasets import make_dataloaders
from src.models.build import build_model
from src.losses import gaussian_nll

cfg = load_config()

train_loader, _, _ = make_dataloaders(cfg=cfg, source="system", batch_size=8)

X, y = next(iter(train_loader))

model = build_model(cfg, input_dim=X.shape[-1])

mu, sigma = model(X)

loss = gaussian_nll(y=y, mu=mu, sigma=sigma)

print("Loss:", float(loss))

Loss: 2334585088.0


In [ ]:
from src.config import load_config
from src.data.datasets import make_dataloaders
from src.models.build import build_model
from src.losses import gaussian_nll

cfg = load_config()

train_loader, _, _ = make_dataloaders(cfg=cfg, source="system", batch_size=8)
X, y = next(iter(train_loader))

model = build_model(cfg, input_dim=X.shape[-1])
mu, sigma = model(X)
print("mu mean/std:", mu.mean().item(), mu.std().item())
print("y  mean/std:", y.mean().item(), y.std().item())
print("sigma mean/min/max:", sigma.mean().item(), sigma.min().item(), sigma.max().item())

loss = gaussian_nll(y=y, mu=mu, sigma=sigma)
print("loss:", loss.item())

mu mean/std: -0.0016384193440899253 0.04943379759788513
y  mean/std: 54046.11328125 12215.3544921875
sigma mean/min/max: 1000.704345703125 1000.6107788085938 1000.7916870117188
loss: 1540.38232421875


In [ ]:
# testing for the transformer.py

import torch
from src.models.backbones.transformer import TransformerBackbone, TransformerBackboneConfig

cfg = TransformerBackboneConfig(
    num_features = 8,
    context_length = 168,
    d_model = 128,
    n_heads = 4,
    n_layers = 4,
    d_ff = 256,
    dropout = 0.1,
    attention_type = "causal",
    positional_encoding_type = "learned",
    pooling = "last",
)

model = TransformerBackbone(cfg)
X = torch.randn(8,168,8)
context = model(X)
print ("Context Shape : ", context.shape)

Context Shape :  torch.Size([8, 128])


C:\Users\Asees\Documents\Project\physics-constrained-prob-load-forecast\src\models\backbones\transformer.py:177: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(


In [ ]:
import torch
from src.models.forecasters.transformer_gaussian import TransformerGaussianForecaster

model = TransformerGaussianForecaster(
    num_features=7,
    context_length=168,
    horizon=24,
    d_model=128,
    n_heads=4,
    n_layers=4,
    d_ff=256,
    dropout=0.1,
    attention_type="causal",
    positional_encoding_type="learned",
    pooling="last",
    head_hidden_dim=128,
    head_dropout=0.1,
    min_sigma=1e-3,
    sigma_activation="softplus",
)

X = torch.randn(8, 168, 7)
out = model(X)

print("mu shape:", out.mu.shape)
print("sigma shape:", out.sigma.shape)
print("sigma min:", out.sigma.min().item())

mu shape: torch.Size([8, 24])
sigma shape: torch.Size([8, 24])
sigma min: 0.27366015315055847


In [ ]:
from src.config import load_config
from src.data.datasets import make_dataloaders
from src.models.build import build_model
from src.losses import gaussian_nll

cfg = load_config()

train_loader, _, _ = make_dataloaders(cfg=cfg, source="system", batch_size=8, return_meta=False)
X, y = next(iter(train_loader))

model = build_model(cfg, input_dim=X.shape[-1])
out = model(X)

print("mu:", out.mu.shape)
print("sigma:", out.sigma.shape)
print("sigma min:", out.sigma.min().item())

loss = gaussian_nll(y=y, mu=out.mu, sigma=out.sigma)
print("loss:", loss.detach().item())

mu: torch.Size([8, 24])
sigma: torch.Size([8, 24])
sigma min: 0.0010000000474974513
loss: 859093585625088.0


C:\Users\Asees\Documents\Project\physics-constrained-prob-load-forecast\src\models\backbones\transformer.py:177: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(


In [ ]:
#Mount drive
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/Othercomputers/Razer/physics-constrained-prob-load-forecast')

print(os.chdir)
!ls


Mounted at /content/drive
<built-in function chdir>
configs  docs		 README.md  requirements.txt  scripts  test_run.ipynb
data	 pyproject.toml  reports    runs	      src      tests


In [ ]:
!python -m src.training.trainer

/content/drive/Othercomputers/Razer/physics-constrained-prob-load-forecast/src/models/backbones/transformer.py:177: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(
[scaling] saved scalers -> /content/drive/Othercomputers/Razer/physics-constrained-prob-load-forecast/runs/exp_transformer_gaussian_v0/scalers
[train][e001 step0000050] nll=0.8114 mae(MW)=4733.50
[train][e001 step0000100] nll=0.5356 mae(MW)=3446.41
[train][e001 step0000150] nll=0.3499 mae(MW)=2786.09
[train][e001 step0000200] nll=0.4372 mae(MW)=3079.81
[train][e001 step0000250] nll=0.2906 mae(MW)=2740.78
[train][e001 step0000300] nll=0.3500 mae(MW)=3043.64
[train][e001 step0000350] nll=0.2654 mae(MW)=2701.15
[train][e001 step0000400] nll=0.3692 mae(MW)=2872.12
[train][e001 step0000450] nll=0.2406 mae(MW)=2584.93
[train][e001 step0000500] nll=0.1618 mae(MW)=2414.45
[train][e001 step0000550] nll=0.2308 mae(MW)=2578.

In [ ]:
import torch
from src.physics.stats import PhysicsStats
from src.physics.penalties import compute_physics_penalty

class Dummy:
    class Bounds:
        enabled = True
        lambda_ = 0.1
        upper_bound = {"strategy": "train_max_margin", "margin": 0.05}
    class Ramp:
        enabled = True
        lambda_ = 0.1
        strategy = "quantile"
        quantile = 0.99
    class Smooth:
        enabled = False
        lambda_ = 0.01

    bounds = Bounds()
    ramp = Ramp()
    smoothness = Smooth()

mu_mw = torch.tensor([
    [70000., 71000., 73000., 76000., 78000.],
    [65000., 65200., 65400., 70000., 76000.]
])

stats = PhysicsStats(train_max_mw=75000.0, ramp_limit_mw=2500.0)

out = compute_physics_penalty(mu_mw=mu_mw, cfg=Dummy(), stats=stats)

print("total:", out.total.item())
print("bounds:", out.bounds.item())
print("ramp:", out.ramp.item())
print("smooth:", out.smoothness.item())

total: 211375.0
bounds: 0.0
ramp: 211375.0
smooth: 0.0


In [1]:
#Mount drive
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/Othercomputers/Razer/physics-constrained-prob-load-forecast')

print(os.chdir)
!ls

Mounted at /content/drive
<built-in function chdir>
configs  docs		 README.md  requirements.txt  scripts  test_run.ipynb
data	 pyproject.toml  reports    runs	      src      tests


In [2]:
!python -m src.training.trainer --source zones

/content/drive/Othercomputers/Razer/physics-constrained-prob-load-forecast/src/models/backbones/transformer.py:177: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(
[scaling] saved scalers -> /content/drive/Othercomputers/Razer/physics-constrained-prob-load-forecast/runs/exp_transformer_physics_zones_v0/scalers
[physics] train_max_mw=85464.12, ramp_limit_mw=6765.13
[train][e001 step0000050] forecast_nll=0.8103 physics=0.0015 total=0.8118 mae(MW)=4800.10 bounds=0.0000 ramp=0.0015 smooth=0.0000
[train][e001 step0000100] forecast_nll=0.5437 physics=0.0005 total=0.5443 mae(MW)=3488.83 bounds=0.0000 ramp=0.0005 smooth=0.0000
[train][e001 step0000150] forecast_nll=0.4216 physics=0.0006 total=0.4222 mae(MW)=3068.48 bounds=0.0000 ramp=0.0006 smooth=0.0000
[train][e001 step0000200] forecast_nll=0.3159 physics=0.0002 total=0.3161 mae(MW)=2808.76 bounds=0.0000 ramp=0.0002 smooth=0.0000
